# Луна поёт своим голосом

Блокнот для Google Colab. Делает две вещи:

1. **Один раз** обучает модель её голоса (RVC, через Applio) на записях из папки `studio/reference` проекта Luna.
2. **Для каждой песни** отделяет вокал (Demucs), заменяет его её голосом и склеивает с музыкой. На выходе готовый mp3, который добавляется в Luna кнопкой **«Добавить кавер её голосом...»** (Настройки → Песни).

**Перед началом:** Среда выполнения → Сменить тип среды → **GPU (T4)**. Без GPU обучение не закончится.

Голос Луны синтетический (edge-tts), а не голос реального человека, поэтому подмена идёт на её собственный голос. Для каверов используй песни, которые тебе можно слушать; готовый mp3 остаётся у тебя.

Блокнот собран по актуальной инструкции Applio, но я не мог запустить его в Colab: если какая-то ячейка упадёт, пришли мне текст ошибки.

## Шаг 1. Установка (около 5–8 минут)
Если среда перезапустится, повтори эту ячейку.

In [ ]:
# @title Установка Applio и Google Drive (модель сохранится на Диске)
from google.colab import drive
try:
    drive.mount("/content/drive")
except Exception as e:
    print("Диск не подключён, модель будет жить только в этой сессии:", e)

%cd /content
!git config --global advice.detachedHead false
!git clone https://github.com/IAHispano/Applio
%cd /content/Applio
!apt update -y
!apt install -y portaudio19-dev
!curl -LsSf https://astral.sh/uv/install.sh | sh
!uv venv .venv --python 3.12 --seed
!uv pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match --python /content/Applio/.venv/bin/python
!uv pip install -q matplotlib-inline websockets --python /content/Applio/.venv/bin/python
!/content/Applio/.venv/bin/python core.py prerequisites --models --pretraineds-hifigan --exe
!cp assets/config_template.json assets/config.json
print("Готово: Applio установлен.")

## Шаг 2. Обучение её голоса (один раз, около 20–40 минут)

Если модель уже обучена и лежит на Диске в `MyDrive/LunaVoice`, пропусти этот шаг и сразу запусти ячейку «Загрузить готовую модель».

Нужны записи её голоса: на компьютере с Luna выполни `py studio/make_reference.py` (нужен интернет, один раз) и загрузи **все mp3 из `studio/reference`** в ячейке ниже. Пустые файлы (0 байт) блокнот пропустит сам.

In [ ]:
# @title Загрузить записи её голоса (mp3 из studio/reference)
import os, shutil
from google.colab import files
DATASET = "/content/luna_dataset"
shutil.rmtree(DATASET, ignore_errors=True)
os.makedirs(DATASET)
uploaded = files.upload()
kept = 0
for name, data in uploaded.items():
    if len(data) > 1024 and name.lower().endswith((".mp3", ".wav", ".flac", ".ogg", ".m4a")):
        with open(os.path.join(DATASET, os.path.basename(name)), "wb") as f:
            f.write(data)
        kept += 1
print("Принято записей:", kept, "(пустые и не аудио пропущены)")
assert kept >= 3, "Нужно хотя бы 3-4 записи; запусти make_reference.py и загрузи всё из studio/reference"

In [ ]:
# @title Обучить модель Luna
import shlex
MODEL = "Luna"
SR = 40000
TOTAL_EPOCH = 300      # для 7-10 минут записей хватает; если голос дрожит, поставь 400-500
BATCH = 8
PY = "/content/Applio/.venv/bin/python"
%cd /content/Applio
!{shlex.join([PY, "core.py", "preprocess", "--model-name", MODEL, "--dataset-path", DATASET, "--sample-rate", str(SR), "--cpu-cores", "2", "--cut-preprocess", "Automatic", "--chunk-len", "3.0", "--overlap-len", "0.3", "--normalization-mode", "none"])}
!{shlex.join([PY, "core.py", "extract", "--model-name", MODEL, "--f0-method", "rmvpe", "--sample-rate", str(SR), "--cpu-cores", "2", "--gpu", "0", "--embedder-model", "contentvec", "--include-mutes", "2"])}
!{shlex.join([PY, "core.py", "index", "--model-name", MODEL, "--index-algorithm", "Auto"])}
!{shlex.join([PY, "core.py", "train", "--model-name", MODEL, "--save-every-epoch", "50", "--save-only-latest", "--total-epoch", str(TOTAL_EPOCH), "--sample-rate", str(SR), "--batch-size", str(BATCH), "--gpu", "0", "--vocoder", "HiFi-GAN", "--pretrained"])}

In [ ]:
# @title Сохранить модель на Диск (чтобы не обучать заново)
import glob, os, shutil
MODEL = "Luna"
logs = f"/content/Applio/logs/{MODEL}"
pths = sorted(glob.glob(f"{logs}/{MODEL}_*e_*s.pth") + glob.glob(f"{logs}/{MODEL}.pth"), key=os.path.getmtime)
idx = sorted(glob.glob(f"{logs}/*.index"), key=os.path.getmtime)
assert pths, "Не нашёл файл весов в " + logs + ": обучение не закончилось?"
if os.path.isdir("/content/drive/MyDrive"):
    dest = "/content/drive/MyDrive/LunaVoice"
    os.makedirs(dest, exist_ok=True)
    shutil.copy(pths[-1], dest + "/Luna.pth")
    if idx:
        shutil.copy(idx[-1], dest + "/Luna.index")
    print("Сохранено в", dest)
else:
    print("Диск не подключён: модель останется до конца этой сессии")
print("Веса:", pths[-1], "| индекс:", idx[-1] if idx else "нет")

In [ ]:
# @title (Только если модель уже обучена) Загрузить готовую модель с Диска
import os, shutil
src = "/content/drive/MyDrive/LunaVoice"
dst = "/content/Applio/logs/Luna"
os.makedirs(dst, exist_ok=True)
assert os.path.isfile(src + "/Luna.pth"), "На Диске нет LunaVoice/Luna.pth: обучи модель в шаге 2"
shutil.copy(src + "/Luna.pth", dst + "/Luna.pth")
if os.path.isfile(src + "/Luna.index"):
    shutil.copy(src + "/Luna.index", dst + "/Luna.index")
print("Модель на месте")

## Шаг 3. Кавер на песню (по 5–10 минут на песню)
Этот шаг повторяешь для каждой песни. Установка Demucs нужна один раз за сессию.

In [ ]:
# @title Установка Demucs (отделяет вокал от музыки)
%cd /content
!uv venv .venv_demucs --python 3.11 --seed
!uv pip install -q "numpy<2" "torch==2.5.1" "torchaudio==2.5.1" --extra-index-url https://download.pytorch.org/whl/cu121 --python /content/.venv_demucs/bin/python
!uv pip install -q demucs soundfile --python /content/.venv_demucs/bin/python
print("Demucs готов")

In [ ]:
# @title Загрузить песню и отделить вокал
import os, glob, shutil
from google.colab import files
shutil.rmtree("/content/song_in", ignore_errors=True)
shutil.rmtree("/content/sep", ignore_errors=True)
os.makedirs("/content/song_in")
up = files.upload()
assert len(up) == 1, "Загрузи ровно один файл песни"
name = list(up)[0]
song_path = "/content/song_in/" + os.path.basename(name)
open(song_path, "wb").write(up[name])
SONG_TITLE = os.path.splitext(os.path.basename(name))[0]
!/content/.venv_demucs/bin/python -m demucs --two-stems=vocals -n htdemucs -o /content/sep "{song_path}"
vocals = glob.glob("/content/sep/htdemucs/*/vocals.wav")
music = glob.glob("/content/sep/htdemucs/*/no_vocals.wav")
assert vocals and music, "Demucs не выдал вокал и музыку: пришли мне текст ошибки выше"
VOCALS, MUSIC = vocals[0], music[0]
print("Вокал:", VOCALS, "| музыка:", MUSIC)

In [ ]:
# @title Заменить вокал её голосом
# Высота: если пел мужчина, поставь 12 (на октаву выше, под её голос); если пела женщина, 0.
PITCH = 12  # @param {type:"slider", min:-12, max:12, step:1}
INDEX_RATE = 0.5  # @param {type:"slider", min:0, max:1, step:0.1}
PROTECT = 0.33  # @param {type:"slider", min:0, max:0.5, step:0.01}
import glob, os, shlex
logs = "/content/Applio/logs/Luna"
pths = sorted(glob.glob(f"{logs}/Luna_*e_*s.pth") + glob.glob(f"{logs}/Luna.pth"), key=os.path.getmtime)
idx = sorted(glob.glob(f"{logs}/*.index"), key=os.path.getmtime)
assert pths, "Модель не найдена: сначала шаг 2"
PY = "/content/Applio/.venv/bin/python"
%cd /content/Applio
cmd = [PY, "core.py", "infer", "--input-path", VOCALS, "--output-path", "/content/vocals_luna.wav", "--pth-path", pths[-1],
       "--index-path", idx[-1] if idx else "", "--pitch", str(PITCH), "--index-rate", str(INDEX_RATE), "--protect", str(PROTECT),
       "--f0-method", "rmvpe", "--export-format", "WAV", "--embedder-model", "contentvec"]
!{shlex.join(cmd)}
assert os.path.isfile("/content/vocals_luna.wav"), "Замена голоса не удалась: пришли мне текст ошибки выше"
print("Её вокал готов")

In [ ]:
# @title Собрать песню и скачать
VOICE_VOLUME = 1.0  # @param {type:"slider", min:0.5, max:2.0, step:0.1}
MUSIC_VOLUME = 0.9  # @param {type:"slider", min:0.3, max:1.5, step:0.1}
import os, re
from google.colab import files
safe = re.sub(r"[^\w \-]", "", SONG_TITLE).strip() or "song"
out = f"/content/{safe} (Luna).mp3"
flt = (f"[0:a]aresample=44100,volume={VOICE_VOLUME}[v];[1:a]aresample=44100,volume={MUSIC_VOLUME}[m];"
       "[v][m]amix=inputs=2:duration=longest:dropout_transition=0,volume=2")
!ffmpeg -y -loglevel error -i /content/vocals_luna.wav -i "{MUSIC}" -filter_complex "{flt}" -b:a 192k "{out}"
assert os.path.isfile(out), "ffmpeg не собрал файл"
print("Готово:", out)
files.download(out)

## Дальше в Luna

Настройки → **Песни** → **«Добавить кавер её голосом...»** → выбери скачанный mp3. Попроси: «спой <название>». Она исполнит запись и будет знать, что это её голос.

Если голос звучит нечётко: переобучи с `TOTAL_EPOCH = 500` или добавь больше записей (`py studio/make_reference.py`), а если женский вокал поётся слишком низко или высоко, поменяй `PITCH`.